# Extended CNN training

Seven 1D-CNN variants test different assumptions about biological time courses. Model selection, classification reports and confusion matrices use only the fixed `validate` fold (called *tune* here); `test` is not loaded in this notebook.

## Contents and architecture map

The links point directly to the corresponding notebook section. The schemas show the main information path; every encoder is followed by a regularized dense classification head.

| Section | Inductive bias | Compact schema | Saved artifact |
|---|---|---|---|
| [First-Difference](#variant-first-difference) | Levels and local slopes are complementary | `x → CNN ┐`<br>`Δx → CNN ┴→ concat → head` | `extended_cnn_first_difference` |
| [FFT](#variant-fft) | Global frequency content complements time-local features | `x, Δx, log |rFFT(x)| → CNNs → concat → head` | `extended_cnn_fft` |
| [Gated fusion](#variant-gated-fusion) | Learn per-feature reliance on raw values versus derivatives | `f = g·f_raw + (1−g)·f_Δ → head` | `extended_cnn_gated_fusion` |
| [Multi-scale CNN](#variant-multi-scale) | Short and long motifs should be detected in parallel | `Conv(k=3/7/15) → concat → head` | `extended_cnn_multi_scale` |
| [TCN](#variant-tcn) | Dilations provide a large temporal receptive field | `dilated residual blocks d=1/2/4 → GAP → head` | `extended_cnn_tcn` |
| [Shift-robust CNN](#variant-shift-robust) | Small temporal shifts should affect predictions less | `stride-1 CNN → global average pooling → head` | `extended_cnn_shift_robust` |
| [Ordinal classification](#variant-ordinal) | Dose classes have a known order | `CNN → latent dose score → ordered centers → logits` | `extended_cnn_ordinal` |
| [Direct comparison](#direct-comparison) | Select by tune macro-F1 and tune loss | metrics and parameter count | — |
| [History comparison](#training-history-comparison) | Diagnose under/overfitting across all variants | four synchronized panels | — |

<a id="setup"></a>
## 1. Setup and fixed folds

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from functools import partial

from pytorch_timecourse_classification.analysis import (
    analyze_training_history, evaluate_classifier, plot_history_comparison,
)
from pytorch_timecourse_classification.data import load_training_folds
from pytorch_timecourse_classification.experiments import (
    create_model, fit_model, save_and_verify_experiment,
)
from pytorch_timecourse_classification.models.extended_cnns import ExtendedCNNClassifier
from pytorch_timecourse_classification.training import TrainingConfig

In [ ]:
train_fold, tune_fold, preprocessor = load_training_folds()
class_labels = np.arange(len(preprocessor.class_names))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for fold_name, fold in (("train", train_fold), ("tune", tune_fold)):
    counts = np.bincount(fold.targets.numpy(), minlength=len(class_labels))
    print(f"{fold_name:>5}: {tuple(fold.features.shape)}, {dict(zip(preprocessor.class_names, counts))}")
print(f"device: {device}")

In [ ]:
build_model = partial(
    create_model,
    ExtendedCNNClassifier,
    input_length=train_fold.features.shape[-1],
    num_classes=len(preprocessor.class_names),
    random_seed=42,
)
fit = partial(
    fit_model,
    training_fold=train_fold,
    tuning_fold=tune_fold,
    device=device,
)
analyze_history = analyze_training_history
evaluate_tune = partial(
    evaluate_classifier,
    fold=tune_fold,
    class_names=preprocessor.class_names,
    device=device,
)
save_and_verify = partial(
    save_and_verify_experiment,
    preprocessor=preprocessor,
    training_fold=train_fold,
    tuning_fold=tune_fold,
    device=device,
)

COMMON_MODEL_CONFIG = {
    "branch_channels": (24, 48, 64),
    "kernel_sizes": (9, 7, 5),
    "dilations": (1, 2, 4),
    "hidden_dims": (128, 32),
    "dropout": 0.30,
    "adaptive_pool_size": 4,
}
training_config = partial(
    TrainingConfig,
    epochs=140,
    batch_size=256,
    learning_rate=5e-4,
    weight_decay=5e-4,
    scheduler_step_size=15,
    learning_rate_decay=0.80,
    patience=20,
    class_balance_strategy="weighted_loss",
)

<a id="variant-first-difference"></a>
## 2. First-Difference CNN

The raw branch learns absolute signal levels and shapes; the derivative branch sees `Δx[t] = x[t+1] − x[t]` and emphasizes rises, falls and turning points. Concatenation preserves both views instead of forcing one representation to replace the other.

```text
raw x ─────────→ residual/dilated CNN ─┐
Δx = x[t+1]-x[t] → residual/dilated CNN ─┴→ concatenate → MLP → dose
```

### 2.1 Model generation

In [ ]:
first_difference_model, first_difference_parameter_count = build_model(
    architecture="first_difference", **COMMON_MODEL_CONFIG
)
first_difference_training_config = training_config()

### 2.2 Training

In [ ]:
first_difference_history = fit(
    first_difference_model, first_difference_training_config
)

### 2.3 History analysis

This is the reference for the extensions below. Parallel high train/tune losses suggest underfitting; a falling train loss with deteriorating tune loss suggests overfitting.

In [ ]:
first_difference_summary = analyze_history(first_difference_history)

### 2.4 Tune confusion analysis

In [ ]:
first_difference_tune_metrics = evaluate_tune(first_difference_model)

### 2.5 Saving artifacts

In [ ]:
first_difference_reloaded, first_difference_paths = save_and_verify(
    "extended_cnn_first_difference", first_difference_model, first_difference_history
)

<a id="variant-fft"></a>
## 3. FFT CNN

A third branch receives `log1p(abs(rFFT(x)))`. It directly exposes slow versus fast variation and periodicity, but magnitude discards phase and temporal localization. Keeping the raw and derivative branches prevents the spectrum from becoming the only view.

```text
x ───────────────→ CNN ─┐
Δx ──────────────→ CNN ─┼→ concatenate → MLP → dose
log(1+|rFFT(x)|) → CNN ─┘
```

### 3.1 Model generation

In [ ]:
fft_model, fft_parameter_count = build_model(
    architecture="fft", **{**COMMON_MODEL_CONFIG, "dropout": 0.35}
)
fft_training_config = training_config(
    class_balance_strategy="balanced_sampler", learning_rate=4e-4, weight_decay=7e-4
)

### 3.2 Training

In [ ]:
fft_history = fit(fft_model, fft_training_config)

### 3.3 History analysis

The FFT branch is useful only if tune macro-F1 or tune loss improves. A train-only gain means the extra representation mostly added variance.

In [ ]:
fft_summary = analyze_history(fft_history)

### 3.4 Tune confusion analysis

In [ ]:
fft_tune_metrics = evaluate_tune(fft_model)

### 3.5 Saving artifacts

In [ ]:
fft_reloaded, fft_paths = save_and_verify(
    "extended_cnn_fft", fft_model, fft_history
)

<a id="variant-gated-fusion"></a>
## 4. Gated raw/derivative fusion

Raw and derivative encoders produce equally sized feature vectors. A learned sigmoid gate chooses continuously, feature by feature, how much of each representation to retain. This is more selective than unconditional concatenation, but the gate can overfit or collapse to one branch.

```text
f_raw ─┐                 ┌→ g·f_raw
       ├→ sigmoid gate g ┤          ├→ sum → MLP → dose
f_Δ ───┘                 └→ (1-g)·f_Δ
```

### 4.1 Model generation

In [ ]:
gated_model, gated_parameter_count = build_model(
    architecture="gated_fusion", **COMMON_MODEL_CONFIG
)
gated_training_config = training_config(weight_decay=7e-4)

### 4.2 Training

In [ ]:
gated_history = fit(gated_model, gated_training_config)

### 4.3 History analysis

Compare against First-Difference: a lower parameter count or smaller train–tune gap with similar macro-F1 supports selective fusion; a rapid train-only gain suggests an overly flexible gate.

In [ ]:
gated_summary = analyze_history(gated_history)

### 4.4 Tune confusion analysis

In [ ]:
gated_tune_metrics = evaluate_tune(gated_model)

### 4.5 Saving artifacts

In [ ]:
gated_reloaded, gated_paths = save_and_verify(
    "extended_cnn_gated_fusion", gated_model, gated_history
)

<a id="variant-multi-scale"></a>
## 5. Multi-Scale CNN

Parallel kernels inspect the same trajectory at short, medium and long scales. This avoids forcing one sequential stack to learn every receptive field, and is attractive when sharp transitions and broad response envelopes are both informative.

```text
         ┌→ Conv k=3  → GAP ─┐
x ──────┼→ Conv k=7  → GAP ─┼→ concatenate → MLP → dose
         └→ Conv k=15 → GAP ─┘
```

### 5.1 Model generation

In [ ]:
multi_scale_model, multi_scale_parameter_count = build_model(
    architecture="multi_scale",
    **{**COMMON_MODEL_CONFIG, "multi_scale_kernel_sizes": (3, 7, 15)},
)
multi_scale_training_config = training_config(learning_rate=6e-4)

### 5.2 Training

In [ ]:
multi_scale_history = fit(
    multi_scale_model, multi_scale_training_config
)

### 5.3 History analysis

A competitive tune score with fewer parameters would support the explicit scale prior. Underfitting suggests that one convolution per scale is too shallow rather than that multi-scale processing is unhelpful.

In [ ]:
multi_scale_summary = analyze_history(multi_scale_history)

### 5.4 Tune confusion analysis

In [ ]:
multi_scale_tune_metrics = evaluate_tune(multi_scale_model)

### 5.5 Saving artifacts

In [ ]:
multi_scale_reloaded, multi_scale_paths = save_and_verify(
    "extended_cnn_multi_scale", multi_scale_model, multi_scale_history
)

<a id="variant-tcn"></a>
## 6. Temporal Convolutional Network (TCN)

Residual convolutions with exponentially increasing dilation enlarge the receptive field without repeatedly downsampling the trajectory. The TCN can relate distant time points while retaining the optimization advantages of convolutional residual blocks.

```text
x → 1×1 projection → ResConv(d=1) → ResConv(d=2) → ResConv(d=4) → GAP → MLP
```

### 6.1 Model generation

In [ ]:
tcn_model, tcn_parameter_count = build_model(
    architecture="tcn", **{**COMMON_MODEL_CONFIG, "branch_channels": (16, 32, 48)}
)
tcn_training_config = training_config(learning_rate=4e-4, weight_decay=7e-4)

### 6.2 Training

In [ ]:
tcn_history = fit(tcn_model, tcn_training_config)

### 6.3 History analysis

If long-range context matters, the TCN should improve tune performance without needing a very wide dense head. Strong train performance with weak tune performance calls for a narrower TCN or stronger dropout.

In [ ]:
tcn_summary = analyze_history(tcn_history)

### 6.4 Tune confusion analysis

In [ ]:
tcn_tune_metrics = evaluate_tune(tcn_model)

### 6.5 Saving artifacts

In [ ]:
tcn_reloaded, tcn_paths = save_and_verify(
    "extended_cnn_tcn", tcn_model, tcn_history
)

<a id="variant-shift-robust"></a>
## 7. Shift-robust CNN

Stride-one, length-preserving convolutions avoid anchoring features to an early downsampling grid. Global average pooling then summarizes whether a motif occurred, with less dependence on its exact position. This provides approximate—not exact—shift invariance; random temporal-shift augmentation would strengthen it further.

```text
x → same-length Conv blocks (stride 1) → global average over time → MLP → dose
```

### 7.1 Model generation

In [ ]:
shift_robust_model, shift_robust_parameter_count = build_model(
    architecture="shift_robust", **COMMON_MODEL_CONFIG
)
shift_robust_training_config = training_config(learning_rate=6e-4)

### 7.2 Training

In [ ]:
shift_robust_history = fit(
    shift_robust_model, shift_robust_training_config
)

### 7.3 History analysis

Improvement would suggest that exact event timing varies across cells. If performance drops, absolute timing is probably discriminative or global averaging removes too much temporal structure.

In [ ]:
shift_robust_summary = analyze_history(shift_robust_history)

### 7.4 Tune confusion analysis

In [ ]:
shift_robust_tune_metrics = evaluate_tune(shift_robust_model)

### 7.5 Saving artifacts

In [ ]:
shift_robust_reloaded, shift_robust_paths = save_and_verify(
    "extended_cnn_shift_robust", shift_robust_model, shift_robust_history
)

<a id="variant-ordinal"></a>
## 8. Ordinal classification

Dose labels are numerically ordered. The model therefore predicts one latent dose score and compares it with learned class centers that are constrained to remain ordered. Cross-entropy still trains the resulting class logits, so this head works with the shared trainer. The useful bias is that confusing neighboring doses is geometrically easier than jumping across the entire dose range.

```text
x → CNN → scalar score s
ordered centers c₀ < c₁ < … < cₖ → logits[j] = −(s−cⱼ)² / temperature
```

### 8.1 Model generation

In [ ]:
ordinal_model, ordinal_parameter_count = build_model(
    architecture="ordinal", **{**COMMON_MODEL_CONFIG, "ordinal_temperature": 0.8}
)
ordinal_training_config = training_config(learning_rate=4e-4, weight_decay=7e-4)

### 8.2 Training

In [ ]:
ordinal_history = fit(ordinal_model, ordinal_training_config)

### 8.3 History analysis

The ordinal bias is helpful if most residual errors move toward adjacent doses and macro-F1 remains competitive. It can underfit when time-course phenotypes are not monotonic in dose, despite the numerical label order.

In [ ]:
ordinal_summary = analyze_history(ordinal_history)

### 8.4 Tune confusion analysis

In [ ]:
ordinal_tune_metrics = evaluate_tune(ordinal_model)

### 8.5 Saving artifacts

In [ ]:
ordinal_reloaded, ordinal_paths = save_and_verify(
    "extended_cnn_ordinal", ordinal_model, ordinal_history
)

<a id="direct-comparison"></a>
## 9. Direct comparison

Prefer the smallest variant whose tune macro-F1 and tune loss are competitive. Accuracy alone can hide a model that ignores difficult doses. The ordinal model should additionally be inspected for whether its remaining errors are predominantly between neighboring doses.

In [ ]:
comparison = pd.DataFrame([
    {"variant": "first_difference", "parameters": first_difference_parameter_count, **first_difference_summary, **first_difference_tune_metrics},
    {"variant": "fft", "parameters": fft_parameter_count, **fft_summary, **fft_tune_metrics},
    {"variant": "gated_fusion", "parameters": gated_parameter_count, **gated_summary, **gated_tune_metrics},
    {"variant": "multi_scale", "parameters": multi_scale_parameter_count, **multi_scale_summary, **multi_scale_tune_metrics},
    {"variant": "tcn", "parameters": tcn_parameter_count, **tcn_summary, **tcn_tune_metrics},
    {"variant": "shift_robust", "parameters": shift_robust_parameter_count, **shift_robust_summary, **shift_robust_tune_metrics},
    {"variant": "ordinal", "parameters": ordinal_parameter_count, **ordinal_summary, **ordinal_tune_metrics},
]).set_index("variant")
comparison.round(4)

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(13, 4))
comparison[["tune_accuracy", "tune_macro_f1"]].plot.bar(
    ax=axes[0], ylim=(0, 1), rot=35
)
axes[0].set(title="Tune classification metrics", ylabel="score")
comparison["parameters"].plot.bar(ax=axes[1], color="0.4", rot=35)
axes[1].set(title="Trainable parameters", ylabel="parameters")
for axis in axes:
    axis.grid(axis="y", alpha=0.2)
figure.tight_layout()

### Interpretation and next steps

The comparison separates representational assumptions from simple model size. First-Difference is the main reference; FFT tests frequency information, gated fusion tests adaptive representation mixing, Multi-Scale and TCN test temporal context, Shift-Robust tests timing variability, and Ordinal tests whether the dose order is a useful constraint. A variant is convincing only when its tune macro-F1 improves across several seeds without a widening train–tune gap.

For the next iteration, repeat the candidates with several seeds and compare mean and standard deviation. For gated fusion, inspect the gate distribution; for Multi-Scale, ablate individual kernel sizes; for TCN, vary dilation depth; for Shift-Robust, evaluate deliberately shifted tune trajectories; and for Ordinal, report mean absolute class-index error in addition to macro-F1. Keep test untouched until the architecture and hyperparameters are frozen.

<a id="training-history-comparison"></a>
## 10. Training-history comparison

The four panels compare train and tune loss and accuracy for all seven variants. Each dot marks the checkpoint epoch restored after early stopping. This is one combined figure; the earlier history cells are per-model diagnostic plots.

In [ ]:
history_comparison_figure = plot_history_comparison({
    "first_difference": first_difference_history,
    "fft": fft_history,
    "gated_fusion": gated_history,
    "multi_scale": multi_scale_history,
    "tcn": tcn_history,
    "shift_robust": shift_robust_history,
    "ordinal": ordinal_history,
})